## Import some packages

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import integrate
import os

import ADFWI
from ADFWI.model import AcousticModel
from ADFWI.propagator import AcousticPropagator
from ADFWI.survey import Receiver, Source, Survey
from ADFWI.utils import numpy2tensor, wavelet
project_path = "./data"
for subdir in ("model", "waveform", "survey"):
    os.makedirs(os.path.join(project_path, subdir), exist_ok=True)

## Basic Parameter

In [ ]:
device = "npu:0"         # Specify the CPU/GPU/NPU device
dtype = torch.float32     # Set data type to 32-bit floating point
backend = ADFWI.set_backend(device, dtype=dtype)
ox, oz = 0, 0             # Origin coordinates for x and z directions
nz, nx = 120, 1576        # Grid dimensions in z and x directions
dx, dz = 25, 25           # Grid spacing in x and z directions
nt, dt = 3581, 0.001      # Time steps and time interval
nabc = 50                 # Thickness of the absorbing boundary layer
f0 = 15                   # Initial frequency in Hz
free_surface = True       # Enable free surface boundary condition


In [ ]:
def cal_waveform(vp_init):
    device = "npu:0"         # Specify the CPU/GPU/NPU device
    dtype = torch.float32     # Set data type to 32-bit floating point
    backend = ADFWI.set_backend(device, dtype=dtype)
    ox, oz = 0, 0             # Origin coordinates for x and z directions
    nz, nx = 120, 1576        # Grid dimensions in z and x directions
    dx, dz = 25, 25           # Grid spacing in x and z directions
    nt, dt = 3581, 0.001      # Time steps and time interval
    nabc = 50                 # Thickness of the absorbing boundary layer
    f0 = 15                   # Initial frequency in Hz
    free_surface = True       # Enable free surface boundary condition

    # Calculate density (rho) based on velocity
    rho_init = np.power(vp_init, 0.25) * 310

    # Initialize the AcousticModel with parameters and properties
    model = AcousticModel(ox, oz, nx, nz, dx, dz,
                        vp_init, rho_init,
                        vp_grad=False,
                        free_surface=free_surface,
                        abc_type="PML",
                        abc_jerjan_alpha=0.007,
                        nabc=nabc
                    )

    src_data = np.load(os.path.join(project_path,"real-data/src_loc.npz"))["data"]
    rcv_data = np.load(os.path.join(project_path,"real-data/rcv_loc.npz"))["data"]
    # Define source positions in the model
    src_z = src_data[:,1]
    src_x = src_data[:,0]
    src_t, src_v = wavelet(nt, dt, f0, amp0=1)  # Create time and wavelet amplitude
    src_v = integrate.cumtrapz(src_v*-1, axis=-1, initial=0)  # Integrate wavelet to get velocity
    source = Source(nt=nt, dt=dt, f0=f0)  # Initialize source object
    for i in range(len(src_x)):
        source.add_source(src_x=src_x[i], src_z=src_z[i], src_wavelet=src_v, src_type="mt", src_mt=np.array([[1,0,0],[0,1,0],[0,0,1]]))
    # Define receiver positions in the model
    rcv_z = rcv_data[:,1]+1
    rcv_x = rcv_data[:,0]
    receiver = Receiver(nt=nt, dt=dt)  # Initialize receiver object
    for i in range(len(rcv_x)):
        receiver.add_receiver(rcv_x=rcv_x[i], rcv_z=rcv_z[i], rcv_type="pr")
    # Create a survey object using the defined source and receiver
    survey = Survey(source=source, receiver=receiver)

    # Initialize the wave propagator using the specified model and survey configuration
    F = AcousticPropagator(model, survey)

    # Perform the forward propagation to record waveforms
    record_waveform = F.forward()

    # Extract recorded pressure wavefield and particle velocities
    rcv_p = record_waveform["p"]  # Recorded pressure wavefield
    return rcv_p,F.src_x,F.rcv_x

rcv_range = np.load(os.path.join(project_path,"real-data/rcv_range.npz"))["data"]
rcv_mask = np.load(os.path.join(project_path,"real-data/rcv_mask.npz"))["data"]

obs_p = np.load(os.path.join(project_path,"real-data/obs_data.npz"))["data"]
obs_p_masks = np.load(os.path.join(project_path,"real-data/obs_data_mask.npz"))["data"]
obs_p = np.transpose(obs_p, (0, 2, 1))
obs_p_masks = np.transpose(obs_p_masks, (0, 2, 1))

In [ ]:
from ADFWI.utils.first_arrivel_picking import apply_mute
from ADFWI.utils.offset_mute import mute_offset

def normalize(data):
    mask    = torch.sum(torch.abs(data),axis=1,keepdim=True) == 0
    max_val = torch.max(torch.abs(data),axis=1,keepdim=True).values
    max_val = max_val.masked_fill(mask, 1)
    data = data/max_val
    return data

def propose_data(rcv_p,obs_p,rcv_mask,src_x,rcv_x,waveform_mute_offset = 400,waveform_mute_late_window = 0.1):
    # step1: mask the data
    rcv_p = rcv_p.cpu()
    syn_p = torch.zeros((obs_p.shape[0],obs_p.shape[1],obs_p.shape[2]))
    for k in range(rcv_p.shape[0]):
        syn_p[k] = rcv_p[k,...,np.argwhere(rcv_mask[k]).tolist()].squeeze()

    # step2: mute offset
    receiver_masks_2D = numpy2tensor(rcv_mask) 
    syn_p = numpy2tensor(syn_p)
    obs_p = numpy2tensor(obs_p)
    shot_index = np.arange(src_x.shape[0])
    if waveform_mute_offset is not None:
        receiver_mask_2D = receiver_masks_2D[shot_index] # [shot, rcv]
        src_x            = src_x.cpu()[shot_index]
        rcv_x_list       = rcv_x.cpu()
        rcv_x = torch.zeros(syn_p.shape[0],syn_p.shape[-1])
        for i in range(syn_p.shape[0]):
            rcv_x[i] = rcv_x_list[np.argwhere(receiver_mask_2D[i]).tolist()].squeeze()   
        syn_p = mute_offset(rcv_x,src_x,dx,syn_p,waveform_mute_offset)
        obs_p = mute_offset(rcv_x,src_x,dx,obs_p,waveform_mute_offset)
    
    # step3: mute late window
    if waveform_mute_late_window is not None:
        syn_p_temp = syn_p.clone()
        obs_p_temp = obs_p.clone()
        for i in range(syn_p.shape[0]):
            syn_p[i] = apply_mute(waveform_mute_late_window, syn_p_temp[i], dt)
            obs_p[i] = apply_mute(waveform_mute_late_window, obs_p_temp[i], dt)

    # normalize
    syn_p = normalize(syn_p)
    obs_p = normalize(obs_p)    
    return syn_p,obs_p

In [ ]:
def plot2D(syn_p,obs_p,shot=0,sparse=5):
    fig,axs = plt.subplots(1,1,figsize=(12,8))
    # select the trace 
    for i in range(0,syn_p.shape[-1],sparse):
        syn_plot = syn_p[shot,:,i]
        obs_plot = obs_p[shot,:,i]
        axs.plot(obs_plot+i,np.arange(syn_p.shape[1])*dt,c='k',linewidth=0.5)
        axs.plot(syn_plot+i,np.arange(syn_p.shape[1])*dt,c='r',linewidth=0.5)
    axs.set_ylim(0, syn_p.shape[1]*dt)
    axs.invert_yaxis()
    axs.set_ylabel("Times (s)",fontsize=20)
    axs.tick_params(labelsize = 20)
    axs.set_xlabel("Receiver ID",fontsize=20)
    plt.show()

def plot1D(syn_p,obs_p,shot=0,trace=100):
    fig,axs = plt.subplots(1,1,figsize=(8,3))
    obs_plot = obs_p[shot,:,trace]
    syn_plot = syn_p[shot,:,trace]
    axs.plot(np.arange(obs_p.shape[1])*dt,obs_plot,c='k',linewidth=0.5,label='observed')
    axs.plot(np.arange(syn_p.shape[1])*dt,syn_plot,c='r',linewidth=0.5,label='inverted')
    plt.legend(loc='upper right')
    plt.show()

## Initial

In [ ]:
from ADFWI.propagator.gradient_process import smooth2d
# Load the Marmousi model dataset
vp_init = np.load(os.path.join(project_path,"real-data/ref_model1.npz"))["data"].T
vp_init = smooth2d(vp_init,span=10)
rcv_p_init,src_x,rcv_x = cal_waveform(vp_init)

syn_p_init,obs_p_init = propose_data(rcv_p_init,obs_p,rcv_mask,src_x,rcv_x,waveform_mute_offset = 400,waveform_mute_late_window = 0.2)

In [ ]:
plot2D(syn_p_init,obs_p_init,shot=0,sparse=5)
plot1D(syn_p_init,obs_p_init,shot=0,trace=100)

## reference

In [ ]:
# Load the Marmousi model dataset
vp_reference = np.load(os.path.join(project_path,"real-data/ref_model5.npz"))["data"].T
rcv_p_ref,src_x,rcv_x = cal_waveform(vp_reference)

syn_p_ref,obs_p_ref = propose_data(rcv_p_ref,obs_p,rcv_mask,src_x,rcv_x,waveform_mute_offset = 400,waveform_mute_late_window = 0.2)

In [ ]:
plot2D(syn_p_ref,obs_p_ref,shot=0,sparse=5)
plot1D(syn_p_ref,obs_p_ref,shot=0,trace=100)

## SGD

In [ ]:
# Load the Marmousi model dataset
vp_SGD = np.load(os.path.join(project_path,"inversion-step-v5/iter_vp_step3.npz"))["data"][-1]
rcv_p_SGD,src_x,rcv_x = cal_waveform(vp_SGD)

syn_p_SGD,obs_p_SGD = propose_data(rcv_p_SGD,obs_p,rcv_mask,src_x,rcv_x,waveform_mute_offset = 400,waveform_mute_late_window = 0.2)

In [ ]:
plot2D(syn_p_SGD,obs_p_SGD,shot=0,sparse=5)
plot1D(syn_p_SGD,obs_p_SGD,shot=0,trace=100)

## Adam

In [ ]:
# Load the Marmousi model dataset
vp_Adam = np.load(os.path.join(project_path,"inversion-step-v1/iter_vp_step3.npz"))["data"][-1]
rcv_p_Adam,src_x,rcv_x = cal_waveform(vp_Adam)

syn_p_Adam,obs_p_Adam = propose_data(rcv_p_Adam,obs_p,rcv_mask,src_x,rcv_x,waveform_mute_offset = 400,waveform_mute_late_window = 0.2)

In [ ]:
plot2D(syn_p_Adam,obs_p_Adam,shot=0,sparse=5)
plot1D(syn_p_Adam,obs_p_Adam,shot=0,trace=100)

## cmp

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(18,6))
sparse = 5
shot = 0
# select the trace 
for i in range(0,syn_p_ref.shape[-1]//sparse):
    syn_plot = syn_p_ref[shot,:,i*sparse]
    obs_plot = obs_p_ref[shot,:,i*sparse]
    axs[0].plot(obs_plot+i*2,np.arange(syn_p_ref.shape[1])*dt,c='k',linewidth=0.5)
    axs[0].plot(syn_plot+i*2,np.arange(syn_p_ref.shape[1])*dt,c='r',linewidth=0.5)
axs[0].set_ylim(0, syn_p_ref.shape[1]*dt)
axs[0].invert_yaxis()
axs[0].set_ylabel("Times (s)",fontsize=20)
axs[0].tick_params(labelsize = 20)
axs[0].set_xlabel("Receiver ID",fontsize=20)
axs[0].set_title("Reference",fontsize=20)

# select the trace 
for i in range(0,syn_p_ref.shape[-1]//sparse):
    syn_plot = syn_p_SGD[shot,:,i*sparse]
    obs_plot = obs_p_SGD[shot,:,i*sparse]
    axs[1].plot(obs_plot+i*2,np.arange(syn_p_ref.shape[1])*dt,c='k',linewidth=0.5)
    axs[1].plot(syn_plot+i*2,np.arange(syn_p_ref.shape[1])*dt,c='r',linewidth=0.5)
axs[1].set_ylim(0, syn_p_ref.shape[1]*dt)
axs[1].invert_yaxis()
axs[1].set_ylabel("Times (s)",fontsize=20)
axs[1].tick_params(labelsize = 20)
axs[1].set_xlabel("Receiver ID",fontsize=20)
axs[1].set_title("ADFWI",fontsize=20)

plt.show()

In [ ]:
fig,axs = plt.subplots(1,1,figsize=(8,3))
shot = 0
trace = 80
obs_plot        = obs_p_ref[shot,:,trace]
syn_plot_ref    = syn_p_ref[shot,:,trace]
syn_plot_ADFWI  = syn_p_SGD[shot,:,trace]
axs.plot(np.arange(obs_p_ref.shape[1])*dt,obs_plot,c='k',linewidth=0.5,label='observed')
axs.plot(np.arange(obs_p_ref.shape[1])*dt,syn_plot_ref,c='r',linewidth=0.5,label='reference')
axs.plot(np.arange(obs_p_ref.shape[1])*dt,syn_plot_ADFWI,c='b',linewidth=0.5,label='ADFWI')
plt.legend(loc='upper right')
plt.xlim(1,2)
plt.show()

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(18,6))
sparse = 5
shot = 0
# select the trace 
for i in range(0,syn_p_init.shape[-1]//sparse):
    syn_plot = syn_p_init[shot,:,i*sparse]
    obs_plot = obs_p_ref[shot,:,i*sparse]
    axs[0].plot(obs_plot+i*2,np.arange(syn_p_init.shape[1])*dt,c='k',linewidth=0.5)
    axs[0].plot(syn_plot+i*2,np.arange(syn_p_init.shape[1])*dt,c='r',linewidth=0.5)
axs[0].set_ylim(0, syn_p_init.shape[1]*dt)
axs[0].invert_yaxis()
axs[0].set_ylabel("Times (s)",fontsize=20)
axs[0].tick_params(labelsize = 20)
axs[0].set_xlabel("Receiver ID",fontsize=20)
axs[0].set_title("Initial",fontsize=20)

# select the trace 
for i in range(0,syn_p_init.shape[-1]//sparse):
    syn_plot = syn_p_SGD[shot,:,i*sparse]
    obs_plot = obs_p_SGD[shot,:,i*sparse]
    axs[1].plot(obs_plot+i*2,np.arange(syn_p_init.shape[1])*dt,c='k',linewidth=0.5)
    axs[1].plot(syn_plot+i*2,np.arange(syn_p_init.shape[1])*dt,c='r',linewidth=0.5)
axs[1].set_ylim(0, syn_p_init.shape[1]*dt)
axs[1].invert_yaxis()
axs[1].set_ylabel("Times (s)",fontsize=20)
axs[1].tick_params(labelsize = 20)
axs[1].set_xlabel("Receiver ID",fontsize=20)
axs[1].set_title("ADFWI",fontsize=20)

plt.show()

In [ ]:
fig,axs = plt.subplots(1,1,figsize=(8,3))
shot = 0
trace = 180
obs_plot        = obs_p_ref[shot,:,trace]
syn_plot_init   = syn_p_init[shot,:,trace]
syn_plot_ADFWI  = syn_p_SGD[shot,:,trace]
axs.plot(np.arange(obs_p_ref.shape[1])*dt,obs_plot,c='k',linewidth=0.5,label='observed')
axs.plot(np.arange(obs_p_ref.shape[1])*dt,syn_plot_init,c='r',linewidth=0.5,label='Initial')
axs.plot(np.arange(obs_p_ref.shape[1])*dt,syn_plot_ADFWI,c='b',linewidth=0.5,label='ADFWI')
plt.legend(loc='upper right')
plt.xlim(0.5,1.5)
plt.show()